# Fase 1: Análisis de Calidad y Exploración del Dataset de RR.HH.


In [27]:
import pandas as pd

# Mostrar todas las columnas al imprimir DataFrames
pd.set_option('display.max_columns', None)

## 1. Carga de datos y visión general

- Dimensiones: (1474, 35)
- Se ven columnas con tipo `float64` que deberían ser enteras y valores `NaN` en primeras filas.

In [28]:
df = pd.read_csv("hr.csv")
print("Dimensiones:", df.shape)
print("\nPrimeras 5 filas:")
df.head()

Dimensiones: (1474, 35)

Primeras 5 filas:


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41.0,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,sALES eXECUTIVE,4.0,Single,5993.0,19479,8,Y,Yes,11,3,1,80.0,0,8,0.0,1,6,4,0,5.0
1,49.0,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,rESEARCH sCIENTIST,2.0,Married,5130.0,24907,1,Y,No,23,4,4,NaN,1,10,3.0,3,10,7,1,7.0
2,37.0,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,lABORATORY tECHNICIAN,3.0,Single,2090.0,2396,6,Y,Yes,15,3,2,NaN,0,7,3.0,3,0,0,0,0.0
3,33.0,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,rESEARCH sCIENTIST,3.0,Married,2909.0,23159,1,Y,Yes,11,3,3,80.0,0,8,3.0,3,8,7,3,0.0
4,27.0,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,lABORATORY tECHNICIAN,2.0,Married,3468.0,16632,9,Y,No,12,3,4,80.0,1,6,3.0,3,2,2,2,2.0


## 2. Tipos de datos y estructura

- 20 columnas `int64`, 9 `object`, 6 `float64`.
- Columnas `float64` problemáticas (deben ser enteras tras imputar): Age, JobSatisfaction, MonthlyIncome, TrainingTimesLastYear, YearsWithCurrManager, StandardHours.

In [29]:
print("Conteo de tipos de datos:")
print(df.dtypes.value_counts())

print("\nColumnas float64 (deberían ser enteras):")
float_cols = df.select_dtypes(include='float64').columns.tolist()
print(float_cols)

Conteo de tipos de datos:
int64      20
str         9
float64     6
Name: count, dtype: int64

Columnas float64 (deberían ser enteras):
['Age', 'JobSatisfaction', 'MonthlyIncome', 'StandardHours', 'TrainingTimesLastYear', 'YearsWithCurrManager']


## 3. Valores nulos

| Columna                 | Nulos | Porcentaje |
|-------------------------|------:|-----------:|
| StandardHours           | 164   | 11.13%     |
| YearsWithCurrManager    | 148   | 10.04%     |
| MaritalStatus           | 132   | 8.96%      |
| BusinessTravel          | 117   | 7.94%      |
| TrainingTimesLastYear   | 88    | 5.97%      |
| Age                     | 73    | 4.95%      |
| EducationField          | 58    | 3.93%      |
| OverTime                | 44    | 2.99%      |
| Department              | 29    | 1.97%      |
| JobSatisfaction         | 29    | 1.97%      |
| MonthlyIncome           | 14    | 0.95%      |

**Acciones Fase 2:**
- Eliminar `StandardHours` (constante + nulos).
- Imputar numéricas con mediana.
- Imputar categóricas con moda.

In [30]:
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
print("Nulos por columna:")
print(nulos)

print("\nPorcentaje de nulos:")
print((nulos / len(df)) * 100)

Nulos por columna:
StandardHours            164
YearsWithCurrManager     148
MaritalStatus            132
BusinessTravel           117
TrainingTimesLastYear     88
Age                       73
EducationField            58
OverTime                  44
Department                29
JobSatisfaction           29
MonthlyIncome             14
dtype: int64

Porcentaje de nulos:
StandardHours            11.126187
YearsWithCurrManager     10.040706
MaritalStatus             8.955224
BusinessTravel            7.937585
TrainingTimesLastYear     5.970149
Age                       4.952510
EducationField            3.934871
OverTime                  2.985075
Department                1.967436
JobSatisfaction           1.967436
MonthlyIncome             0.949796
dtype: float64


## 4. Filas duplicadas

- Se encontraron 4 filas duplicadas (2 pares exactos). Las copias aparecen en los últimos índices (1470-1473).
- Se eliminarán con `drop_duplicates()`, quedando 1470 filas.

In [31]:
dups = df.duplicated().sum()
print("Filas duplicadas (exactas):", dups)
if dups:
    print("Ejemplo de filas duplicadas:")
    display(df[df.duplicated(keep=False)].sort_index())

Filas duplicadas (exactas): 4
Ejemplo de filas duplicadas:


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
67,45.0,No,Travel_Rarely,1339,Research & Development,7,3,Life Sciences,1,86,2,Male,59,3,3,rESEARCH sCIENTIST,1.0,Divorced,9724.0,18787,2,Y,No,17,3,3,NaN,1,25,2.0,3,1,0,0,0.0
184,53.0,No,Travel_Rarely,1084,Research & Development,13,2,Medical,1,250,4,Female,57,4,2,mANUFACTURING dIRECTOR,1.0,Divorced,4450.0,26250,1,Y,No,11,3,3,NaN,2,5,3.0,3,4,2,1,3.0
1041,28.0,No,Travel_Rarely,866,Sales,5,3,Medical,1,1469,4,Male,84,3,2,sALES eXECUTIVE,1.0,Single,8463.0,23490,0,Y,No,18,3,4,NaN,0,6,4.0,3,5,4,1,NaN
1222,24.0,Yes,Travel_Rarely,240,Human Resources,22,1,Human Resources,1,1714,4,Male,58,1,1,hUMAN rESOURCES,3.0,Married,1555.0,11585,1,Y,No,11,3,3,80.0,1,1,2.0,3,1,0,0,0.0
1470,28.0,No,Travel_Rarely,866,Sales,5,3,Medical,1,1469,4,Male,84,3,2,sALES eXECUTIVE,1.0,Single,8463.0,23490,0,Y,No,18,3,4,NaN,0,6,4.0,3,5,4,1,NaN
1471,53.0,No,Travel_Rarely,1084,Research & Development,13,2,Medical,1,250,4,Female,57,4,2,mANUFACTURING dIRECTOR,1.0,Divorced,4450.0,26250,1,Y,No,11,3,3,NaN,2,5,3.0,3,4,2,1,3.0
1472,24.0,Yes,Travel_Rarely,240,Human Resources,22,1,Human Resources,1,1714,4,Male,58,1,1,hUMAN rESOURCES,3.0,Married,1555.0,11585,1,Y,No,11,3,3,80.0,1,1,2.0,3,1,0,0,0.0
1473,45.0,No,Travel_Rarely,1339,Research & Development,7,3,Life Sciences,1,86,2,Male,59,3,3,rESEARCH sCIENTIST,1.0,Divorced,9724.0,18787,2,Y,No,17,3,3,NaN,1,25,2.0,3,1,0,0,0.0


## 5. Columnas constantes o sin valor útil

| Columna        | Valor | Acción      |
|----------------|-------|-------------|
| EmployeeCount  | 1     | Eliminar    |
| Over18         | Y     | Eliminar    |
| StandardHours  | 80.0  | Eliminar    |

In [32]:
const_cols = []

for col in df.columns:
    unicos = df[col].dropna().unique()
    if len(unicos) == 1:
        const_cols.append((col, unicos[0], df[col].notna().sum()))
        
print("Columnas con un solo valor distinto (ignorando nulos):")
for col, val, freq in const_cols:
    print(f"- {col}: {val} (frecuencia {freq})")

Columnas con un solo valor distinto (ignorando nulos):
- EmployeeCount: 1 (frecuencia 1474)
- Over18: Y (frecuencia 1474)
- StandardHours: 80.0 (frecuencia 1310)


## 6. Estadísticas de variables numéricas clave

| Variable             | Media   | Mediana | Observación                                      |
|----------------------|---------|---------|--------------------------------------------------|
| Age                  | 36.9    | 36      | Plantilla joven                                  |
| MonthlyIncome        | 6497.45 | 4907    | Media inflada por salarios altos (outliers)     |
| YearsAtCompany       | 7.0     | 5       | La mitad lleva menos de 5 años                  |
| TotalWorkingYears    | 11.27   | 10      | Experiencia razonable                            |
| JobSatisfaction      | 2.73    | 3       | Satisfacción ligeramente baja                   |
| WorkLifeBalance      | 2.76    | 3       | Equilibrio mejorable                            |

In [33]:
clave = ['Age', 'MonthlyIncome', 'YearsAtCompany', 'TotalWorkingYears', 
         'JobSatisfaction', 'WorkLifeBalance']
df[clave].describe().round(2)

,Age,MonthlyIncome,YearsAtCompany,TotalWorkingYears,JobSatisfaction,WorkLifeBalance
count,1401.00,1460.00,1474.00,1474.00,1445.00,1474.00
mean,36.94,6497.45,7.00,11.27,2.73,2.76
std,9.11,4706.06,6.12,7.79,1.11,0.71
min,18.00,1009.00,0.00,0.00,1.00,1.00
25%,30.00,2909.00,3.00,6.00,2.00,2.00
50%,36.00,4907.00,5.00,10.00,3.00,3.00
75%,43.00,8377.00,9.00,15.00,4.00,3.00
max,60.00,19999.00,40.00,40.00,4.00,4.00


## 7. Variable objetivo: Attrition (rotación)

- No: 1236 (83.85%)
- Yes: 238 (16.15%)

**Interpretación:** Fuga de talento real (16%). Clases desbalanceadas → necesario considerar en modelado.

In [34]:
print("Conteos absolutos:")
print(df['Attrition'].value_counts())

print("\nPorcentajes:")
print(df['Attrition'].value_counts(normalize=True) * 100)

Conteos absolutos:
Attrition
No     1236
Yes     238
Name: count, dtype: int64

Porcentajes:
Attrition
No     83.85346
Yes    16.14654
Name: proportion, dtype: float64


## 8. Calidad de variables categóricas

- **JobRole:** mayúsculas/minúsculas inconsistentes y espacios al inicio/final. Ejemplo: `' sALES eXECUTIVE '`.  
  - Limpiar con `str.strip().str.title()`.

- **Department:** 29 nulos. Imputar con moda ('Research & Development').

- **MaritalStatus:** typo 'Marreid' (3 casos) debe corregirse a 'Married'. Además 132 nulos → imputar moda ('Married').

- **OverTime:** 44 nulos → imputar moda ('No').

- **BusinessTravel, EducationField, Gender:** sin problemas mayores, solo imputar nulos con moda.

In [35]:
print("JobRole - valores únicos (primeros 5):")
print(df['JobRole'].unique()[:5])

print("\nDepartment - frecuencias con nulos:")
print(df['Department'].value_counts(dropna=False))

print("\nMaritalStatus - frecuencias con nulos:")
print(df['MaritalStatus'].value_counts(dropna=False))

print("\nOverTime - frecuencias con nulos:")
print(df['OverTime'].value_counts(dropna=False))

JobRole - valores únicos (primeros 5):
<StringArray>
[          ' sALES eXECUTIVE ',        ' rESEARCH sCIENTIST ',
     ' lABORATORY tECHNICIAN ',    ' mANUFACTURING dIRECTOR ',
 ' hEALTHCARE rEPRESENTATIVE ']
Length: 5, dtype: str

Department - frecuencias con nulos:
Department
Research & Development    941
Sales                     440
Human Resources            64
NaN                        29
Name: count, dtype: int64

MaritalStatus - frecuencias con nulos:
MaritalStatus
Married     604
Single      437
Divorced    298
NaN         132
Marreid       3
Name: count, dtype: int64

OverTime - frecuencias con nulos:
OverTime
No     1025
Yes     405
NaN      44
Name: count, dtype: int64


## 9. Resumen de limpieza para Fase 2

| Problema                              | Acción propuesta                                      |
|---------------------------------------|-------------------------------------------------------|
| Columnas constantes                   | Eliminar `EmployeeCount`, `Over18`, `StandardHours`  |
| Identificador único                   | Eliminar `EmployeeNumber`                            |
| Tipos float64 → int64                  | Convertir tras imputar nulos                         |
| Nulos en numéricas                    | Imputar con **mediana**                              |
| Nulos en categóricas                  | Imputar con **moda**                                 |
| JobRole con mayúsculas y espacios     | Limpiar: `str.strip().str.title()`                   |
| MaritalStatus con typo "Marreid"      | Reemplazar por "Married"                             |
| Filas duplicadas (4)                  | Eliminar con `drop_duplicates()`                     |

A modo ilustrativo, para realizar en Fase 2, después de esta operación, el dataset pasaría de 1474 a 1470 filas.
```python
acciones = {
    "Eliminar columnas constantes": ["EmployeeCount", "Over18", "StandardHours"],
    "Corregir typo en MaritalStatus": "Reemplazar 'Marreid' por 'Married'",
    "Limpiar JobRole": "strip() + title()",
    "Imputar numéricas (mediana)": ["YearsWithCurrManager", "TrainingTimesLastYear", "Age", "JobSatisfaction", "MonthlyIncome"],
    "Imputar categóricas (moda)": ["MaritalStatus", "BusinessTravel", "EducationField", "OverTime", "Department"],
    "Eliminar duplicados": "df.drop_duplicates()"
}
for k, v in acciones.items():
    print(f"{k}: {v}")
```

## 10. Conclusiones generales

- El dataset tiene **1474 empleados** (1470 tras eliminar duplicados) y **35 variables**.
- **Calidad aceptable** pero con nulos moderados en 11 columnas y algunos problemas de formato.
- La **tasa de rotación** es del 16.15%, justificando análisis predictivo.
- Columnas inútiles a eliminar: `EmployeeCount`, `Over18`, `StandardHours`.
- La limpieza permitirá pasar a análisis exploratorio avanzado y modelado.